# 2 · Building and running a flow

Compile nodes + edges into a validated plan, then execute. Execution is **eager and parallel by default**: independent branches run concurrently.

- `GraphNode` — an instance of a registered node type
- `GraphEdge` — wires one node's output handle to another's input handle
- `compile()` — validates structure, topologically sorts, type-checks edges
- `execute()` — async generator that streams events (the primary API)
- `execute_sync()` — blocking wrapper (use it from scripts; in notebooks use `await collect(execute(...))` since the kernel already owns an event loop)

In [ ]:
from typing import Annotated

from conductor import GraphEdge, GraphNode, NodeRegistry, compile
from conductor.execution.engine import collect, execute
from conductor.widgets import Output, Text

registry = NodeRegistry()


@registry.node("echo", version=1, name="Echo", description="Returns input")
def echo(text: Annotated[str, Text(label="Input")]) -> Annotated[str, Output(label="Output")]:
    return text


@registry.node("upper", version=1, name="Uppercase", description="Uppercases")
def upper(text: Annotated[str, Text(label="Input")]) -> Annotated[str, Output(label="Output")]:
    return text.upper()


@registry.node("join", version=1, name="Join", description="Joins two strings")
def join(
    a: Annotated[str, Text(label="A")],
    b: Annotated[str, Text(label="B")],
    separator: Annotated[str, Text(label="Separator")] = " + ",
) -> Annotated[str, Output(label="Result")]:
    return f"{a}{separator}{b}" 

## Build the graph

```text
  echo("hello") ──> upper ──┐
                              ├──> join ──> result
  echo("world") ──────────────┘
```

`n1` and `n2` have no incoming edges — they run in parallel. `n3` waits for `n1`; `n4` waits for both `n3` and `n2`.

In [ ]:
nodes = [
    GraphNode("n1", "echo@1", {"text": "hello"}),
    GraphNode("n2", "echo@1", {"text": "world"}),
    GraphNode("n3", "upper@1", None),  # no static data — input comes from edge
    GraphNode("n4", "join@1", {"separator": " & "}),
]

edges = [
    GraphEdge("e1", "n1", "n3", "result", "text"),   # echo("hello") -> upper
    GraphEdge("e2", "n3", "n4", "result", "a"),      # upper -> join.a
    GraphEdge("e3", "n2", "n4", "result", "b"),      # echo("world") -> join.b
]

compiled = compile(nodes=nodes, edges=edges, registry=registry)
print("Execution order:", compiled.execution_order)

## Run and collect the final results

`collect()` consumes the event stream and returns the final `{node_id: result}` dict — this is what `execute_sync()` does under the hood. In a notebook we `await` it directly.

In [ ]:
results = await collect(execute(compiled))
for node_id, result in results.items():
    print(f"  {node_id}: {result}")

## Streaming execution

Iterate the event stream directly to see what the engine is doing in real time. Notice that `n1` and `n2` both start before either finishes — they are dispatched as independent tasks.

In [ ]:
async for event in execute(compiled):
    match event["type"]:
        case "node_start":
            print(f"  [start]    {event['node_id']}")
        case "node_complete":
            print(f"  [complete] {event['node_id']}: {event['result']}")
        case "flow_complete":
            print(f"  [done]     {event['results']}")
        case _:
            print(f"  [{event['type']}]")